[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Peewee, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)

# A Small Catalog


## What you will be able to do

Put the whole guide together: models behind a `DatabaseProxy` so the backend is a setting, a catalog
loaded in one transaction, a listing page whose query count is asserted rather than hoped for, a
JSON response built from rows rather than from objects, and a search behind one function so that the
part which does not port is in one place. Run a test that points the same models at a database in
memory. And recognize the failure this guide has been building towards: a models module and a
database that no longer agree, which says nothing until a query mentions the column.


## The idea

### The problem

Everything in this guide worked because the database was made from the models a moment earlier. Once
they are made at different times, by different people, they drift, and peewee will not notice. A
field added to a model is a promise about a column, and nothing checks it until a query goes out
with that column's name in it.

That failure arrives at the worst moment: not when the application starts, not when the tests run
against a database built from the current models, but when a request touches the one query that
mentions the new field. The **Migrations** notebook had the answer already, and this notebook is
about making it a habit rather than a rescue.

### What this notebook is

One small application, assembled from the pieces the rest of the guide built, with every claim it
makes printed: how many queries the listing page costs, what the response looks like, what the other
backend would be sent, and what a test of it looks like.

### Why it works that way

A `DatabaseProxy` stands where a `Database` goes and is given a real one later. That is what lets a
models module be imported by an application, a test and a migration tool, each of which wants a
different database, without any of them editing the models.

### Where this shows up

Every project past its first week. The specific arrangement here, a proxy, a URL from the
environment and a search behind a function, is what most small peewee applications end up looking
like.

### What this notebook covers

The models and the proxy. The catalog loaded in one transaction. The listing page, under an asserted
query count. The response, in the three shapes peewee can give a row. Search, and the seam the
backend switch runs through. A test. Then the four failures: drift, an uninitialized proxy, a model
where JSON was expected, and a pool nobody gave a connection back to.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from peewee import CharField, IntegerField, Model, SqliteDatabase

db = SqliteDatabase(":memory:")


class Book(Model):                              # what the models said when the table was made
    title = CharField()
    year = IntegerField()

    class Meta:
        database = db


db.create_tables([Book])
Book.create(title="The Salt Road", year=2014)


class Book(Model):                              # what the models say after somebody adds a field
    title = CharField()
    year = IntegerField()
    shelf = CharField(null=True)

    class Meta:
        database = db
        table_name = "book"


print("the models say:", [field.name for field in Book._meta.sorted_fields])
try:
    print("a listing:", [book.title for book in Book.select()])
except Exception as error:
    print("the database says:", type(error).__name__ + ":", error)
```

```
the models say: ['id', 'title', 'year', 'shelf']
the database says: OperationalError: no such column: t1.shelf
```

Nothing was wrong until a query was run. The model was declared, the application started, the table
was there, and the field that does not exist was a problem only when something asked for it.


## Setup

Fifteen imports, peewee installed and pinned, twelve books, and the models behind a proxy.

- `peewee` is the library, and `Model`, the field classes, `DatabaseProxy`, `SqliteDatabase` and
  `PostgresqlDatabase`, from it, are what the application is written with
- `connect_url` builds the database from a URL, and `os` is where that URL comes from
- `FTS5Model`, `SearchField` and `RowIDField` are the search index, from **FTS5Model and
  SearchField**
- `assert_query_count` proves the listing page's cost, `model_to_dict` builds the response, and
  `MaxConnectionsExceeded` and `PooledSqliteDatabase` are the last of the Common errors
- `chunked` loads the catalog, `json` prints the response, and `re`, `subprocess`, `sys`, `tempfile`
  and `Path` run the test and keep this notebook's output the same on every run
- `version` and `PackageNotFoundError` install peewee 4.5.1 where the version is not that

`database` is a `DatabaseProxy`. The models name it, `configure` gives it a real database built from
`DATABASE_URL`, and nothing else in the application mentions a backend. That one indirection is what
makes the test at the end possible without a second copy of the models.


In [1]:
import json
import os
import re
import subprocess
import sys
import tempfile
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    if version("peewee") != "4.5.1":                                # Colab has 4.4.0, whose wording differs
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "peewee==4.5.1"], check=True)

import peewee
from peewee import (CharField, DatabaseProxy, ForeignKeyField, IntegerField, Model,
                    OperationalError, PostgresqlDatabase, SqliteDatabase, TextField, chunked)
from playhouse.db_url import connect as connect_url
from playhouse.pool import MaxConnectionsExceeded, PooledSqliteDatabase
from playhouse.shortcuts import model_to_dict
from playhouse.sqlite_ext import FTS5Model, RowIDField, SearchField
from playhouse.test_utils import assert_query_count

CATALOG = [                                                         # author, title, year, pages, blurb
    ("Ursula Vance", "The Salt Road", 2014, 312, "A road made of salt, and the sea beside it."),
    ("Ursula Vance", "Nightjar", 2018, 244, "Birds at dusk, and a house nobody lives in."),
    ("Ursula Vance", "The Quiet Engine", 2021, 398, "Engines, quiet ones, and the sea again."),
    ("Marco Pietra", "Stone and Tide", 2009, 501, "Granite, tides, and a quarry above the sea."),
    ("Marco Pietra", "The Lantern Keeper", 2016, 276, "A lantern, a keeper, and long winters."),
    ("Marco Pietra", "Riverwork", 2022, 189, "Rivers, locks and the work of moving water."),
    ("Ines O'Brien", "A Careful Fire", 1998, 420, "Fire, carefully kept, in a cold house."),
    ("Ines O'Brien", "The Long Field", 2004, 355, "One field, many seasons, and the sea far off."),
    ("Ines O'Brien", "Winter Harbour", 2011, 263, "A harbour in winter, and the boats laid up."),
    ("Kofi Mensah", "The Drum Line", 2015, 198, "Drums, a line of them, and a city waking up."),
    ("Kofi Mensah", "Harmattan", 2019, 331, "A dry wind, and what it carries with it."),
    ("Kofi Mensah", "Small Machines", 2023, 287, "Machines, small ones, and the people who mend them."),
]

WORK = Path(tempfile.mkdtemp(prefix="catalog-"))

database = DatabaseProxy()                                          # which database is a setting


class CatalogModel(Model):
    class Meta:
        database = database


class Author(CatalogModel):
    name = CharField(max_length=60, unique=True)


class Book(CatalogModel):
    title = CharField(max_length=80)
    author = ForeignKeyField(Author, backref="books")
    year = IntegerField(index=True)
    pages = IntegerField()
    blurb = TextField()


class BookIndex(FTS5Model):
    rowid = RowIDField()
    title = SearchField()
    blurb = SearchField()

    class Meta:
        database = database
        options = {"content": Book}


def configure(url=None):
    """Point the catalog at a database. The URL is the only line that names a backend."""
    chosen = connect_url(url or os.environ.get("DATABASE_URL", f"sqlite:///{WORK / 'catalog.db'}"))
    database.initialize(chosen)
    return chosen


live = configure()
print("peewee", peewee.__version__, "| backend:", type(live).__name__)
print("models bound:", [model.__name__ for model in (Author, Book, BookIndex)])


peewee 4.5.1 | backend: SqliteDatabase
models bound: ['Author', 'Book', 'BookIndex']


## Worked examples

### The tables, and the catalog in one transaction

Twelve books, four authors, and one `atomic` block around all of it, so that a failure halfway
leaves an empty catalog rather than half of one:


In [2]:
def load(rows):
    """Create the tables and load the catalog, or leave nothing behind."""
    live.create_tables([Author, Book, BookIndex])
    with live.atomic():
        names = sorted({author for author, *_ in rows})
        Author.insert_many([{"name": name} for name in names]).execute()
        keys = {author.name: author.id for author in Author.select()}
        books = [{"title": title, "author": keys[author], "year": year,
                  "pages": pages, "blurb": blurb}
                 for author, title, year, pages, blurb in rows]
        for piece in chunked(books, 100):
            Book.insert_many(piece).execute()
    BookIndex.rebuild()
    return Author.select().count(), Book.select().count()


print("loaded:", load(CATALOG), "(authors, books)")
print("indexed:", BookIndex.select().count())


loaded: (4, 12) (authors, books)
indexed: 12


`insert_many` in chunks, inside one `atomic`, is the shape from **Creating and Changing Rows**. The
index is rebuilt at the end, because with `content` pointing at `book` it holds no text of its own
and has to be told when the text changed.

### The listing page, with its cost asserted

A page shows books with their authors, which is exactly the query that quietly costs one round trip
per row. The count is written into the code so that it cannot drift:


In [3]:
def listing(page=1, per_page=5, since=2000):
    """One page of the catalog, oldest first, with each book's author."""
    query = (Book.select(Book, Author)
                 .join(Author)
                 .where(Book.year >= since)
                 .order_by(Book.year, Book.id)
                 .paginate(page, per_page))
    return [(row.year, row.title, row.author.name) for row in query]


with assert_query_count(1):
    rows = listing()

for year, title, author in rows:
    print(f"  {year}  {title:<20} {author}")


  2004  The Long Field       Ines O'Brien
  2009  Stone and Tide       Marco Pietra
  2011  Winter Harbour       Ines O'Brien
  2014  The Salt Road        Ursula Vance
  2015  The Drum Line        Kofi Mensah


One query for five books and their authors. `select(Book, Author)` is what makes it one, and
`assert_query_count` is what keeps it one: drop the `Author` from that `select` and this cell fails
rather than getting slower.

### The response

A page does not return model instances, it returns data. peewee gives three shapes, and they are
not interchangeable:


In [4]:
query = Book.select(Book.title, Book.year).where(Book.year >= 2020).order_by(Book.title)

print("objects:", [type(row).__name__ for row in query])
print("dicts:  ", list(query.dicts()))
print("tuples: ", list(query.tuples()))


objects: ['Book', 'Book', 'Book']
dicts:   [{'title': 'Riverwork', 'year': 2022}, {'title': 'Small Machines', 'year': 2023}, {'title': 'The Quiet Engine', 'year': 2021}]
tuples:  [('Riverwork', 2022), ('Small Machines', 2023), ('The Quiet Engine', 2021)]


`dicts()` is the one an endpoint wants, because it is already what `json.dumps` takes. `tuples()` is
for a CSV or anything positional. The objects are for code that is going to use them as objects, and
handing one to `json.dumps` is the third of the Common errors below.

When the rows are models rather than a projection, `model_to_dict` converts one:


In [5]:
book = Book.select(Book, Author).join(Author).where(Book.title == "Harmattan").get()

print("everything, and the author too:")
print(" ", json.dumps(model_to_dict(book))[:96], "...")
print()
print("the key instead of the author:")
print(" ", json.dumps(model_to_dict(book, recurse=False)))
print()
print("just what the page needs:")
print(" ", json.dumps(model_to_dict(book, only=[Book.title, Book.year, Book.pages])))


everything, and the author too:
  {"id": 11, "title": "Harmattan", "author": {"id": 2, "name": "Kofi Mensah"}, "year": 2019, "page ...

the key instead of the author:
  {"id": 11, "title": "Harmattan", "author": 2, "year": 2019, "pages": 331, "blurb": "A dry wind, and what it carries with it."}

just what the page needs:
  {"title": "Harmattan", "year": 2019, "pages": 331}


`recurse=False` is the one to reach for by default. Without it, one book's response includes its
author's whole row, and in a model with more relationships it walks further than you meant.

### Search, and the seam

The search is FTS5, which is SQLite's and does not port. Putting it behind one function is what
keeps that fact in one place:


In [6]:
def search(typed, limit=5):
    """Search the catalog for whatever was typed. FTS5 here; another backend needs its own."""
    if isinstance(live, SqliteDatabase):
        query = BookIndex.web_query(typed or "")
        if not query.strip():
            return []
        return list(Book.select(Book, Author)
                        .join(Author)
                        .switch(Book)
                        .join(BookIndex, on=(Book.id == BookIndex.rowid))
                        .where(BookIndex.match(query))
                        .order_by(BookIndex.bm25())
                        .limit(limit))
    raise NotImplementedError("PostgreSQL: a TSVectorField and a GIN index, from SQLite and "
                              "PostgreSQL")


for typed in ("sea", "c++", "engines OR drums", ""):
    found = search(typed)
    print(f"  {typed!r:<20} {[book.title for book in found]}")


  'sea'                ['The Quiet Engine', 'Stone and Tide', 'The Long Field', 'The Salt Road']
  'c++'                []
  'engines OR drums'   ['The Quiet Engine', 'The Drum Line']
  ''                   []


The `NotImplementedError` is deliberate and is the honest shape of this. A search written on FTS5
does not become a PostgreSQL search by changing a URL, and pretending otherwise is how a backend
switch turns into a week. One function, two implementations, and the reader of this code can see
which one they have.

### What the other backend would be sent

No server is running here, and the models still compile for one:


In [7]:
other = PostgresqlDatabase(None)

with other.bind_ctx([Author, Book]):
    print("create:", Book._schema._create_table().query()[0][:92], "...")
    insert = Book.insert(title="x", author=1, year=2024, pages=100, blurb="b")
    print("insert:", " ".join(insert.sql()[0].split())[:92], "...")


create: CREATE TABLE IF NOT EXISTS "book" ("id" SERIAL NOT NULL PRIMARY KEY, "title" VARCHAR(80) NOT ...
insert: INSERT INTO "book" ("title", "author_id", "year", "pages", "blurb") VALUES (%s, %s, %s, %s,  ...


`SERIAL`, `%s` and a `RETURNING` clause, exactly as in **SQLite and PostgreSQL**, from a database
that has never spoken to a server. Everything in this application except `search` would move.

### A test

The proxy is what makes this short: the test points the same models at a database in memory and
never touches the application's:


In [8]:
(WORK / "test_catalog.py").write_text(
    "# A test of the listing page, against a database of its own.\n"
    "import pytest\n"
    "from peewee import SqliteDatabase\n"
    "\n"
    "import catalog\n"
    "\n"
    "\n"
    "@pytest.fixture\n"
    "def catalog_db():\n"
    "    # a database in memory, holding two books, for one test\n"
    "    made = SqliteDatabase(':memory:', pragmas={'foreign_keys': 1})\n"
    "    with made.bind_ctx([catalog.Author, catalog.Book]):\n"
    "        made.create_tables([catalog.Author, catalog.Book])\n"
    "        writer = catalog.Author.create(name='Ursula Vance')\n"
    "        catalog.Book.create(title='The Salt Road', author=writer, year=2014,\n"
    "                            pages=312, blurb='b')\n"
    "        catalog.Book.create(title='Nightjar', author=writer, year=2018,\n"
    "                            pages=244, blurb='b')\n"
    "        yield made\n"
    "\n"
    "\n"
    "def test_listing_shows_only_recent_books(catalog_db):\n"
    "    assert catalog.listing(since=2015) == [(2018, 'Nightjar', 'Ursula Vance')]\n")

print("wrote", (WORK / "test_catalog.py").name)


wrote test_catalog.py


The application's own module has to exist for the test to import, so here it is written out from the
same definitions this notebook has been using:


In [9]:
(WORK / "catalog.py").write_text("""
from peewee import (CharField, DatabaseProxy, ForeignKeyField, IntegerField, Model, TextField)

database = DatabaseProxy()


class CatalogModel(Model):
    class Meta:
        database = database


class Author(CatalogModel):
    name = CharField(max_length=60, unique=True)


class Book(CatalogModel):
    title = CharField(max_length=80)
    author = ForeignKeyField(Author, backref="books")
    year = IntegerField(index=True)
    pages = IntegerField()
    blurb = TextField()


def listing(page=1, per_page=5, since=2000):
    query = (Book.select(Book, Author)
                 .join(Author)
                 .where(Book.year >= since)
                 .order_by(Book.year, Book.id)
                 .paginate(page, per_page))
    return [(row.year, row.title, row.author.name) for row in query]
""")

done = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no",
                       "-p", "no:cacheprovider", "test_catalog.py"],
                      cwd=WORK, capture_output=True, text=True)
print(re.sub(r"in \d+\.\d+s", "in <time>", done.stdout.strip().splitlines()[-1]))


1 passed in <time>


The fixture never mentions a URL, a file or the application's database. `bind_ctx` points the models
at one for the length of the test and puts them back, which is the same call **SQLite and
PostgreSQL** used to compile for another backend.

### When to reach for which

| What the application needs | What it uses |
|---|---|
| the backend as a setting | `DatabaseProxy` plus `connect_url(os.environ[...])` |
| a catalog loaded, or not at all | `insert_many` in `chunked` pieces inside `atomic` |
| a page of rows with their parent | `.select(Book, Author).join(Author)` |
| that page's cost kept honest | `assert_query_count(1)` around it |
| a JSON response | `.dicts()`, or `model_to_dict(row, recurse=False)` |
| positional rows | `.tuples()` |
| a search | one function, with the backend's own implementation inside it |
| a test database | `bind_ctx` onto `SqliteDatabase(":memory:")` |
| to know the schema matches | `pwmigrate app.db status`, from **Migrations** |

### The catalog, finished

Every piece above, as the small application it was all for:


In [10]:
def respond(path, **parameters):
    """The whole application: three routes, each returning data rather than objects."""
    if path == "/books":
        return [{"year": year, "title": title, "author": author}
                for year, title, author in listing(**parameters)]
    if path == "/search":
        return [model_to_dict(book, only=[Book.title, Book.year], recurse=False)
                for book in search(parameters.get("q", ""))]
    if path == "/books/count":
        return {"books": Book.select().count(), "authors": Author.select().count()}
    return {"error": "no such path"}


for path, parameters in (("/books", {"per_page": 2}),
                         ("/search", {"q": "sea"}),
                         ("/books/count", {}),
                         ("/nowhere", {})):
    print(f"  {path:<14} {json.dumps(respond(path, **parameters))[:78]}")


  /books         [{"year": 2004, "title": "The Long Field", "author": "Ines O'Brien"}, {"year":
  /search        [{"title": "The Quiet Engine", "year": 2021}, {"title": "Stone and Tide", "yea
  /books/count   {"books": 12, "authors": 4}
  /nowhere       {"error": "no such path"}


Three routes, no model instance leaving any of them, one query for the listing, and a search that
says plainly which backend it belongs to.

### Where each part came from

| In the application | What it relies on | The notebook that showed it |
|---|---|---|
| `DatabaseProxy` and `connect_url` | a backend chosen by a setting | **SQLite and PostgreSQL** |
| `insert_many` in `chunked` pieces | many rows without meeting the value limit | **Creating and Changing Rows** |
| `with live.atomic()` | a load that is all or nothing | **Transactions** |
| `.select(Book, Author).join(Author)` | a page in one query | **Relationships** |
| `assert_query_count(1)` | a cost that fails rather than drifts | **Relationships** |
| `.paginate(page, per_page)` | a page a reader can jump to | **Selecting Rows** |
| `BookIndex.web_query(typed)` | a search box that is not a query language | **FTS5Model and SearchField** |
| `.dicts()` and `model_to_dict` | rows rather than objects | this notebook |
| `bind_ctx` in the test | the same models, another database | **SQLite and PostgreSQL** |
| `pwmigrate status` beside it | a schema that matches the models | **Migrations** |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/12-a-small-catalog-solutions.ipynb).

**1.** Add a `/authors` route returning each author's name and how many books they have, in one
query, and assert that count.


In [11]:
# your code here


**2.** Return the second page of the listing and show that no title appears on both pages.


In [12]:
# your code here


**3.** Build the same response three ways, with `dicts()`, with `tuples()` and with
`model_to_dict`, and say which you would return from an endpoint.


In [13]:
# your code here


**4.** Search for something with a character FTS5 treats as syntax, and show the route returning a
list rather than raising.


In [14]:
# your code here


**5.** Point the models at a database in memory with `bind_ctx`, load two books into it, and show
the application's own catalog is untouched.


In [15]:
# your code here


**6.** Print what the `book` table and one insert would be on PostgreSQL, without connecting.


In [16]:
# your code here


## Common errors

### peewee.OperationalError: no such column: t1.shelf


In [17]:
class BookLater(CatalogModel):
    """The models, a week later. The database was not told."""

    title = CharField(max_length=80)
    author = ForeignKeyField(Author, backref="later")
    year = IntegerField()
    pages = IntegerField()
    blurb = TextField()
    shelf = CharField(max_length=10, null=True)                     # added to the model only

    class Meta:
        table_name = "book"


list(BookLater.select())


OperationalError: no such column: t1.shelf

The application started, the table was there, and this is the first query that mentioned the new
column. Everything between the deploy and this request looked fine.

What makes it the last error in this guide is that it has a check, and the check runs before anybody
is affected. The **Migrations** notebook's `status` exits non zero on a database that has not caught
up, and its `diff` names the column:


In [18]:
missing = set(field.column_name for field in BookLater._meta.sorted_fields)
present = set(column.name for column in live.get_columns("book"))

print("the model wants:", sorted(missing))
print("the table has:  ", sorted(present))
print("drift:", sorted(missing - present), "<- what a schema diff reports before a request does")


the model wants: ['author_id', 'blurb', 'id', 'pages', 'shelf', 'title', 'year']
the table has:   ['author_id', 'blurb', 'id', 'pages', 'title', 'year']
drift: ['shelf'] <- what a schema diff reports before a request does


### AttributeError: Cannot use uninitialized Proxy.


In [19]:
spare = DatabaseProxy()                                             # never given a real database


class Orphan(Model):
    name = CharField(max_length=40)

    class Meta:
        database = spare


Orphan.create(name="nowhere to go")


AttributeError: Cannot use uninitialized Proxy.

A proxy is a placeholder, and until something calls `initialize` on it there is nothing behind it.
This is the cost of the indirection that makes the backend a setting: the models import cleanly and
fail on first use, so the mistake is a missing line in start up rather than anything visible in the
models.

The fix is that `configure` has to be called, once, before anything queries:


In [20]:
spare.initialize(SqliteDatabase(":memory:"))
spare.create_tables([Orphan])
print("written:", Orphan.create(name="somewhere").name)
print("the catalog's proxy points at:", type(database.obj).__name__)


written: somewhere
the catalog's proxy points at: SqliteDatabase


### TypeError: Object of type Book is not JSON serializable


In [21]:
json.dumps(list(Book.select().limit(2)))


TypeError: Object of type Book is not JSON serializable

A `Book` is an object with fields, not a mapping, and `json.dumps` has no way to guess which of its
attributes are data. The error is clear, and it arrives at the point of serializing rather than at
the point of querying, which in a web application means inside a request rather than at start up.

Three ways out, and the first is the one to reach for:


In [22]:
print("dicts():      ", json.dumps(list(Book.select(Book.title, Book.year).limit(2).dicts())))
print("model_to_dict:", json.dumps([model_to_dict(book, only=[Book.title], recurse=False)
                                    for book in Book.select().limit(2)]))
print("by hand:      ", json.dumps([{"title": book.title} for book in Book.select().limit(2)]))


dicts():       [{"title": "The Salt Road", "year": 2014}, {"title": "Nightjar", "year": 2018}]
model_to_dict: [{"title": "The Salt Road"}, {"title": "Nightjar"}]
by hand:       [{"title": "The Salt Road"}, {"title": "Nightjar"}]


### MaxConnectionsExceeded: Exceeded maximum connections.


In [23]:
import threading

small = PooledSqliteDatabase(str(WORK / "pool.db"), max_connections=1)
small.connect()                                                     # taken, and not given back

def borrow():
    try:
        small.connect()
        print("  the second thread got one")
    except MaxConnectionsExceeded as error:
        print("  MaxConnectionsExceeded:", error)

worker = threading.Thread(target=borrow)
worker.start()
worker.join()


  MaxConnectionsExceeded: Exceeded maximum connections.


One connection, held by the first thread, and the second cannot have one. Nothing is broken: the
pool did what a pool with a limit of one does.

In an application this is a connection that was never returned, usually because a request failed
between taking one and closing it. `close` in a `finally`, or the context manager that does it for
you, is the answer, and the limit is what turns a leak into an error rather than into a slow crawl:


In [24]:
small.close()                                                       # given back

with small.connection_context():                                    # taken and returned for you
    print("  inside the block, idle:", len(small._connections))
print("  after the block, idle: ", len(small._connections))
small.dispose()


  inside the block, idle: 0
  after the block, idle:  1


## Recap

- A models module behind a `DatabaseProxy` can be pointed at a database by an application, a test
  and a migration tool, each of which wants a different one.
- The backend is a URL in the environment. Everything except the search moves with it, which is why
  the search is one function rather than a query in the middle of a page.
- Load with `insert_many` in `chunked` pieces inside one `atomic`, so a failure leaves nothing.
- A listing page costs one query when it selects both models, and `assert_query_count` is what keeps
  it that way as the code changes.
- Return data rather than objects: `.dicts()` for JSON, `.tuples()` for anything positional, and
  `model_to_dict(row, recurse=False)` when the row is a whole model.
- Test by binding the same models to a database in memory with `bind_ctx`.
- The failure this guide ends on is drift: a model and a table that no longer agree, which says
  nothing until a query names the column, and which a schema check catches before a request does.


## What is next

That is the guide. You can write peewee models and read the SQL they become, write and change rows
and know which call sends what, build filters that mean what they read, keep several writes together
in a transaction, load related rows without a query per row, put a document in a column, search text,
migrate a schema that has data in it, and move the whole thing to another backend knowing what will
not come with it.

Where to go next depends on what you want. **asyncpg and psycopg3, Deep Dive** stands a real
PostgreSQL server up and works below the mapper. **sqlite3, Deep Dive** owns the database this guide
ran on: its locking model, FTS5 itself, and what a file can and cannot do. **SQLAlchemy, Deep Dive**
is the other answer to the same problem, and **Why Peewee** said where it wins. And **Testing and
Packaging** takes the test at the end of this notebook and makes it a suite.


---

&#8592; **Previous:** [SQLite and PostgreSQL](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/11-sqlite-and-postgresql.ipynb)  &nbsp;·&nbsp;  [Peewee, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)
